In [ ]:
# Célula 1: motor, config e corpus
REPO <- "https://raw.githubusercontent.com/vicleme/PI3-FFV/main/"                    # endereço Raw do repositório
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor06.R")   # funções do curso
download.file(paste0(REPO, "estrutura/codigo/config.R"), "config.R")                # traz o config.R
source("config.R")                                                                   # cfg: decisões do grupo
cp     <- read.csv(paste0(REPO, "estrutura/banco-de-dados/corpus.csv"),              # corpus congelado
                   stringsAsFactors = FALSE, fileEncoding = "UTF-8")
docs   <- setNames(cp$texto, cp$id)                                                  # vetor nomeado id -> texto
origem <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds"))))   # de que verbete veio cada documento
ix     <- montar(docs, cfg)                                                          # índice, TF-IDF, BM25
estado()                                                                             # estado do R

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-11-25 11:02 · R version 4.6.1 (2026-06-24) · motor06 v2 2026-10-08 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg$minimo` | numeric | 1 | 200 |
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | translit |
| `cfg$stopwords` | character | 203 | a, ao, aos, aquela, aquelas, aquele, ... |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$limiar` | numeric | 1 | 2 |
| `cfg$juizes` | character | 1 | consenso |
| `cfg$grupo` | character | 1 | FFV |
| `cp` | data.frame | 63 × 2 | colunas: id, texto |
| `docs` | character, nomeado | 63 | Santos_p1 Santos é um município brasi..., Santos_p2 Fundada em 1546 por Brás Cu..., Santos_p3 Cidade mais populosa do lit..., Santos_p4 O principal cartão-postal d..., Santos_p5 Verificam-se relatos a resp..., Santos_p6 Em 1543, com o término da c..., ... |
| `ix$toke

In [ ]:
# Célula 2: gabarito e consultas
arqs <- c("qrels_consenso.csv")                                                      # um arquivo só: voto majoritário
for (a in arqs) download.file(paste0(REPO, "estrutura/banco-de-dados/", a), a)       # traz o qrels
qrels <- ler_qrels(arqs, juiz = cfg$juizes)                                          # lê o gabarito
nec   <- read.csv(paste0(REPO, "estrutura/banco-de-dados/necessidades.csv"),         # necessidades, com coluna conjunto
                  stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
consultas <- setNames(nec$texto_consulta, nec$consulta)                              # q01 -> texto digitado
R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))                              # relevantes por consulta
usar <- names(qrels)[R_q > 0]                                                        # consultas avaliadas
names(qrels)[R_q == 0]                                                               # as que ficam fora (esperado: nenhuma)
limpar("Guarujá Cubatão", "translit")                                                # teste dos acentos

character(0)

[1] "guaruja cubatao"

In [ ]:
rocchio <- function(q, Dr, Dnr = character(0), ix, a = 1, b = 0.75, g = 0.15) {  # q: vetor; Dr, Dnr: nomes
  cr  <- rowMeans(ix$w[, Dr, drop = FALSE])                                # centroide dos relevantes
  cnr <- if (length(Dnr)) rowMeans(ix$w[, Dnr, drop = FALSE]) else 0       # dos nao relevantes; sem nenhum, 0
  a * q + b * cr - g * cnr                                                 # a formula, termo a termo
}                                                                          # fim da funcao
ranking_vetor <- function(q, ix)                  # q: vetor de pesos no espaco de ix$vocab
  sort(apply(ix$w, 2, function(d) cosseno(q, d)), # um cosseno por coluna
       decreasing = TRUE)                         # do maior ao menor

In [ ]:
ix_aula <- montar(docs_aula(), cfg_aula())                                  # o indice do curso: so para o teste
q_aula  <- vetor_consulta(tokenizar("modelo de recuperacao"), ix_aula$vocab, ix_aula$idf_tfidf)  # a consulta do curso
names(ranking_vetor(rocchio(q_aula, c("d2", "d3"), "d4", ix_aula), ix_aula))  # tem que dar d3 d2 d1 d6 d8 d5 d7 d4

[1] "d3" "d2" "d1" "d6" "d8" "d5" "d7" "d4"

In [ ]:
q    <- "q07"                                                      # consulta de trabalho: cubatao virou municipio ano
g    <- qrels[[q]]                                                 # os graus dela
q0   <- vetor_consulta(preparar(consultas[[q]], cfg), ix$vocab, ix$idf_tfidf)  # o vetor, com a limpeza do grupo
base <- ranking_vetor(q0, ix)                                      # o mesmo que ranking_cosseno(consultas[[q]], ix, cfg)
Dr   <- intersect(names(g)[g >= cfg$limiar], names(docs))          # os relevantes julgados que estao no corpus
Dnr  <- intersect(names(base)[1:5], names(g)[g == 0])              # grau 0 visto no top-5 base
Dr; Dnr                                                            # confiram antes de seguir

[1] "Cubatao_p8" "Cubatao_p9"

[1] "Santos_p15"  "Cubatao_p20" "Guaruja_p20" "Cubatao_p1"

In [6]:
round(base[1:5], 3)        # os cinco primeiros do ranking base, com o cosseno de cada um
g[names(base)[1:5]]        # os graus que o gabarito deu a esses cinco: g[...] busca no vetor g pelos ids

Santos_p15 Cubatao_p20 Guaruja_p20  Cubatao_p1  Cubatao_p8 
      0.140       0.100       0.096       0.090       0.078

Santos_p15 Cubatao_p20 Guaruja_p20  Cubatao_p1  Cubatao_p8 
          0           0           0           0           2

In [7]:
novo <- ranking_vetor(rocchio(q0, Dr, Dnr, ix), ix)                # a consulta reformulada, buscada de novo
round(rbind(base    = metricas(base, g, cfg$limiar),               # antes...
            rocchio = metricas(novo, g, cfg$limiar)), 3)           # ...e depois, na colecao completa

,P_at_k,AP,RR,nDCG_bin,nDCG_grad
base,0.000,0.267,0.2,0.456,0.456
rocchio,0.667,1.000,1.0,1.000,1.000


In [8]:
names(novo)[1:5]

[1] "Cubatao_p8"  "Cubatao_p9"  "Cubatao_p7"  "Cubatao_p11" "Cubatao_p18"

In [9]:
residual <- function(rk, g, fora, limiar)                               # metricas so no que NAO foi marcado
  metricas(rk[!names(rk) %in% fora], g[!names(g) %in% fora], limiar)    # tira 'fora' do ranking e do gabarito
fora <- c(Dr, Dnr)                                                       # tudo que o usuario marcou
round(rbind(base    = residual(base, g, fora, cfg$limiar),               # base, sem os marcados
            rocchio = residual(novo, g, fora, cfg$limiar)), 3)           # Rocchio, sem os marcados

,P_at_k,AP,RR,nDCG_bin,nDCG_grad
base,0,NA,0,NA,NA
rocchio,0,NA,0,NA,NA


In [10]:
Dr_v   <- intersect(names(base)[1:5], Dr)                               # so os relevantes que estavam no top-5
novo_v <- ranking_vetor(rocchio(q0, Dr_v, Dnr, ix), ix)                 # Rocchio com o que foi visto
fora_v <- c(Dr_v, Dnr)                                                   # os marcados, agora
round(rbind(base    = residual(base,   g, fora_v, cfg$limiar),           # o resto, antes
            rocchio = residual(novo_v, g, fora_v, cfg$limiar)), 3)       # o resto, depois: achou algo NOVO?

,P_at_k,AP,RR,nDCG_bin,nDCG_grad
base,0.333,1.0,1.0,1.000,1.000
rocchio,0.333,0.5,0.5,0.631,0.631


In [11]:
Dr_v

[1] "Cubatao_p8"

In [12]:
names(novo_v)[!names(novo_v) %in% fora_v][1:3]    # os 3 primeiros do Rocchio sem os marcados: ! e %in% tiram os marcados

[1] "Cubatao_p7" "Cubatao_p9" "Santos_p6"

In [13]:
docs[["Cubatao_p7"]]
docs[["Cubatao_p9"]]
docs[["Cubatao_p6"]]

[1] "Em 1803, o Governador da Capitania, Antonio José de Franca e Horta, numa política de povoamento e desenvolvimento da Capitania, ordena a edificação do Povoado de Cubatão, na extinta Fazenda dos Jesuítas, entre os rios Rio Capivari e Rio Santana, na margem direita do Rio Cubatão."

[1] "Com o desenvolvimento da região, foi construída, em 1947, a Via Anchieta, chamada de orgulho da engenharia rodoviária nacional, a mais moderna das rodovias brasileiras na época; com o crescimento, o fortalecimento comercial e industrial, o bairro começou a perder suas características agrícolas e começou a sonhar com o futuro. Organizou-se, em 1948, um grupo de trabalho para lutar pela elevação de Cubatão à categoria de município, que ficou conhecido como Os Emancipadores. A emancipação política e autonomia de Cubatão aconteceria em 9 de abril de 1949."

[1] "Em 1556, durante o Governo Geral, foi doada a Antonio Rodrigues de Almeida uma sesmaria cujas terras partem do Rio das Pedras, até o Rio Pilões; essa sesmaria foi adquirida em 1643 pelos padres da Companhia de Jesus, sendo o embrião da Fazenda Geral dos Jesuítas, também chamada de Fazenda Geral do Cubatão. As terras jesuíticas margeavam o Rio Cubatão, o que lhes dava controle sobre a navegação fluvial que ligava a escarpa da Serra do Mar e o Porto de Santos."

In [15]:
prf <- function(q0, k, ix, p = c(a = 1, b = 0.75, g = 0.15)) {   # pseudo-realimentacao: top-k viram Dr
  top <- names(ranking_vetor(q0, ix))[1:k]                        # os k primeiros do ranking base
  ranking_vetor(rocchio(q0, top, ix = ix,                         # Rocchio sem Dnr...
                        a = p[["a"]], b = p[["b"]], g = p[["g"]]), ix)  # ...com os pesos de p; busca de novo
}                                                                 # fim da funcao

ap_q <- function(txt, grau, k, ix, cfg) {                                  # AP de uma consulta; k = 0: sem PRF
  q0 <- vetor_consulta(preparar(txt, cfg), ix$vocab, ix$idf_tfidf)         # o vetor da consulta
  rk <- if (k == 0) ranking_vetor(q0, ix) else prf(q0, k, ix)              # base ou PRF
  metricas(rk, grau, cfg$limiar)[["AP"]]                                   # so o AP, contra os graus dela
}                                                                          # fim da funcao

In [16]:
dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])      # aqui se escolhe k
tab_dev <- sapply(dev, function(q) {                                # para cada consulta dev...
  a <- function(k) ap_q(consultas[[q]], qrels[[q]], k, ix, cfg)     # o AP desta consulta com um k
  c(base = a(0), k2 = a(2), k3 = a(3), k5 = a(5))                   # ...o AP em cada opcao
})                                                                  # fim do sapply
round(tab_dev, 3)                                                   # uma coluna por consulta
round(rowMeans(tab_dev), 3)                                         # o MAP de cada opcao

,q02,q03,q05,q06,q07,q09,q10
base,0.333,1,0.143,1.0,0.267,0.500,1
k2,0.100,1,0.077,1.0,0.121,0.667,1
k3,0.500,1,0.091,0.5,0.112,0.667,1
k5,0.200,1,0.077,0.5,0.216,0.700,1


base    k2    k3    k5 
0.606 0.566 0.553 0.528

In [17]:
colnames(tab_dev)[tab_dev["k3", ] < tab_dev["base", ]]                    # consultas em que k = 3 piorou
qrels[["q06"]][names(ranking_cosseno(consultas[["q06"]], ix, cfg))[1:5]]  # graus do top-5 base da q06

[1] "q05" "q06" "q07"

Santos_p22  Santos_p23 Cubatao_p17  Santos_p11  Santos_p18 
          2           0           0           0           0

In [21]:
docs[["Cubatao_p17"]]                                                     # o texto do documento que entrou em Dr com k = 3
names(prf(vetor_consulta(preparar(consultas[["q06"]], cfg), ix$vocab, ix$idf_tfidf), 3, ix))[1:3]   # os 3 primeiros da q06 com PRF k = 3

[1] "Em 2008, foi eleita a professora Marcia Rosa, tendo, como vice, o ex-vereador Arlindo Fagundes Filho. Vereadora nas duas últimas legislaturas, Marcia Rosa foi eleita com mais de 40 mil votos, a melhor votação da história de Cubatão, e assumiu a prefeitura em 2009. Marcia foi reeleita na eleição de 2012, tendo como vice o atual vereador Donizete com mais de 39 mil votos. Porém, o TRE-SP cassou o mandato da então Prefeita Marcia Rosa e de seu vice Donizete Tavares no dia 26 de maio de 2014, por fazerem propaganda eleitoral antecipada. O então presidente da câmara de vereadores, Wagner Moura, assumiu a prefeitura tornando-se o novo prefeito da cidade."

[1] "Cubatao_p17" "Santos_p22"  "Santos_p23"

In [22]:
source("config.R")                                                         # recarrega com os campos novos
cfg$prf_k; cfg$rocchio                                                     # confiram com a decisao
ranking_motor <- function(txt, ix, cfg) {                                  # o ranking do motor, com a decisao
  q0 <- vetor_consulta(preparar(txt, cfg), ix$vocab, ix$idf_tfidf)         # o vetor da consulta
  if (cfg$prf_k == 0) ranking_vetor(q0, ix) else prf(q0, cfg$prf_k, ix, cfg$rocchio)  # sem ou com PRF
}                                                                          # fim da funcao
round(head(ranking_motor(consultas[["q07"]], ix, cfg), 5), 3)              # o top-5 que o motor daria na q07

[1] 0

a    b    g 
1.00 0.75 0.15

Santos_p15 Cubatao_p20 Guaruja_p20  Cubatao_p1  Cubatao_p8 
      0.140       0.100       0.096       0.090       0.078

In [23]:
q0_06 <- vetor_consulta(preparar(consultas[["q06"]], cfg), ix$vocab, ix$idf_tfidf)  # consulta original
top3  <- names(ranking_vetor(q0_06, ix))[1:3]                  # os 3 que a PRF assume como relevantes
q3    <- rocchio(q0_06, top3, ix = ix)                         # consulta reformulada, sem Dnr
entrou <- head(sort(q3[q0_06 == 0], decreasing = TRUE), 10)    # os 10 termos novos de maior peso
q0_06[q0_06 > 0]                                               # os termos originais da consulta
round(entrou, 3)                                               # os termos que entraram
round(ix$w[names(entrou), top3], 3)                            # de qual documento veio cada um

futebol  historia    santos 
4.1431347 3.4499875 0.7091475

marcia  esporte     rosa     vice  assumiu donizete   eleita    surfe 
   4.143    3.107    3.107    3.107    2.072    2.072    2.072    2.072 
 titulos     tres 
   2.072    2.072

,Santos_p22,Santos_p23,Cubatao_p17
marcia,0.000,0.000,16.573
esporte,0.000,12.429,0.000
rosa,0.000,0.000,12.429
vice,0.000,0.000,12.429
assumiu,0.000,0.000,8.286
donizete,0.000,0.000,8.286
eleita,0.000,0.000,8.286
surfe,0.000,8.286,0.000
titulos,8.286,0.000,0.000
tres,8.286,0.000,0.000


In [25]:
"jogador" %in% ix$vocab

[1] FALSE

In [26]:
rbind(original = round(q0_06[q0_06 > 0], 3))    # termos originais, em tabela
rbind(entrou   = round(entrou, 3))              # termos novos, em tabela
round(ix$w[names(entrou), top3], 3)             # peso de cada termo em cada documento

,futebol,historia,santos
original,4.143,3.45,0.709


,marcia,esporte,rosa,vice,assumiu,donizete,eleita,surfe,titulos,tres
entrou,4.143,3.107,3.107,3.107,2.072,2.072,2.072,2.072,2.072,2.072


,Santos_p22,Santos_p23,Cubatao_p17
marcia,0.000,0.000,16.573
esporte,0.000,12.429,0.000
rosa,0.000,0.000,12.429
vice,0.000,0.000,12.429
assumiu,0.000,0.000,8.286
donizete,0.000,0.000,8.286
eleita,0.000,0.000,8.286
surfe,0.000,8.286,0.000
titulos,8.286,0.000,0.000
tres,8.286,0.000,0.000


In [27]:
anexar_estado("00_FICHA_PROJETO.md")

Estado do R anexado a 00_FICHA_PROJETO.md (75 linhas de objetos).



In [28]:
anexar_estado("aula06_parteD_consolidado.md")

Estado do R anexado a aula06_parteD_consolidado.md (75 linhas de objetos).

